In [2]:
import cv2
import numpy as np
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

class ChessboardDistanceMeasurer:
    def __init__(self, root):
        self.root = root
        self.root.title("Misuratore Distanze Scacchiera")
        
        # Variabili
        self.points = []
        self.img_path = ""
        self.img_height = 0
        self.img_width = 0
        
        # Frame principale
        self.main_frame = tk.Frame(root)
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Pulsante per caricare l'immagine
        self.load_btn = tk.Button(self.main_frame, text="Carica Immagine", command=self.load_image)
        self.load_btn.pack(pady=10)
        
        # Canvas per visualizzare l'immagine
        self.canvas = tk.Canvas(self.main_frame, bg='gray')
        self.canvas.pack(fill=tk.BOTH, expand=True)
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        
        # Pulsanti di controllo
        self.control_frame = tk.Frame(self.main_frame)
        self.control_frame.pack(pady=10)
        
        self.reset_btn = tk.Button(self.control_frame, text="Reset", command=self.reset, state=tk.DISABLED)
        self.reset_btn.pack(side=tk.LEFT, padx=5)
        
        self.quit_btn = tk.Button(self.control_frame, text="Esci", command=root.quit)
        self.quit_btn.pack(side=tk.LEFT, padx=5)
        
        # Etichetta per le istruzioni
        self.instructions = tk.Label(self.main_frame, text="Seleziona 4 incroci della scacchiera cliccando sull'immagine", 
                                    fg="blue")
        self.instructions.pack(pady=5)
        
        # Etichetta per i risultati
        self.result_label = tk.Label(self.main_frame, text="", fg="green")
        self.result_label.pack(pady=5)
        
    def load_image(self):
        # Apri file dialog per selezionare l'immagine
        self.img_path = filedialog.askopenfilename(
            title="Seleziona immagine scacchiera",
            filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp")]
        )
        
        if not self.img_path:
            return
            
        # Carica l'immagine con OpenCV
        self.cv_img = cv2.imread(self.img_path)
        if self.cv_img is None:
            messagebox.showerror("Errore", "Impossibile caricare l'immagine")
            return
            
        # Converti per visualizzazione in Tkinter
        self.img_height, self.img_width = self.cv_img.shape[:2]
        self.display_image()
        
        # Abilita il reset
        self.reset_btn.config(state=tk.NORMAL)
        self.instructions.config(text="Clicca su 4 incroci della scacchiera")
        
    def display_image(self):
        # Converti da BGR (OpenCV) a RGB (PIL/Tkinter)
        img_rgb = cv2.cvtColor(self.cv_img, cv2.COLOR_BGR2RGB)
        img_pil = Image.fromarray(img_rgb)
        
        # Ridimensiona se troppo grande per lo schermo
        max_size = (800, 600)
        if img_pil.width > max_size[0] or img_pil.height > max_size[1]:
            img_pil.thumbnail(max_size, Image.LANCZOS)
        
        self.tk_img = ImageTk.PhotoImage(img_pil)
        
        # Aggiorna canvas
        self.canvas.delete("all")
        self.canvas.config(width=self.tk_img.width(), height=self.tk_img.height())
        self.canvas.create_image(0, 0, anchor=tk.NW, image=self.tk_img)
        
        # Disegna l'origine (in alto a sinistra)
        self.canvas.create_oval(0, 0, 10, 10, fill="blue", outline="blue")
        self.canvas.create_text(15, 10, text="Origine (0,0)", anchor=tk.NW, fill="blue")
        
    def on_canvas_click(self, event):
        if not hasattr(self, 'cv_img'):
            return
            
        if len(self.points) >= 4:
            return
            
        # Coordinate rispetto all'origine in alto a sinistra
        x_img = event.x
        y_img = event.y
        
        self.points.append((x_img, y_img))
        
        # Disegna il punto sul canvas (coordinate Tkinter)
        point_size = 5
        self.canvas.create_oval(
            event.x - point_size, event.y - point_size,
            event.x + point_size, event.y + point_size,
            fill="red", outline="red"
        )
        self.canvas.create_text(
            event.x + 15, event.y - 15,
            text=f"{len(self.points)}",
            fill="green", font=('Helvetica', '10', 'bold')
        )
        
        if len(self.points) == 4:
            self.display_coordinates()
            
    def display_coordinates(self):
        result_text = "Coordinate dei punti selezionati:\n"
        
        for i, (x, y) in enumerate(self.points, 1):
            # Aggiungi testo delle coordinate accanto ai punti
            self.canvas.create_text(
                x + 20, y - 20,
                text=f"P{i}: ({x}, {y})",
                fill="cyan", font=('Helvetica', '10', 'bold')
            )
            
            result_text += f"Punto {i}: ({x}, {y})\n"
        
        self.result_label.config(text=result_text)
        self.instructions.config(text="Selezione completata. Premi Reset per ricominciare")
        
    def reset(self):
        self.points = []
        if hasattr(self, 'cv_img'):
            self.display_image()
        self.result_label.config(text="")
        self.instructions.config(text="Clicca su 4 incroci della scacchiera")

if __name__ == "__main__":
    root = tk.Tk()
    app = ChessboardDistanceMeasurer(root)
    root.mainloop()
